# Training Pipeline

Trains the same feature-selection and three regression models for NiCoMn, NiFeTi, and ZnYGe.

## 1. Libraries

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import PolynomialFeatures

## 2. Configuration

In [2]:
# Resolve the project root from the notebook location/current working directory.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks" or not (PROJECT_DIR / "datasets").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "datasets"
MODEL_DIR = PROJECT_DIR / "models"

SYSTEMS = {
    "NiCoMn": {"file": "NiCoMn_dataset_complete.csv", "elements": ["Ni", "Co", "Mn"]},
    "NiFeTi": {"file": "NiFeTi_dataset_complete.csv", "elements": ["Ni", "Fe", "Ti"]},
    "ZnYGe":  {"file": "ZnYGe_dataset_complete.csv",  "elements": ["Zn", "Y", "Ge"]},
}

TRAIN_SIZE = 0.70
VAL_SIZE = 0.10
TEST_SIZE = 0.20
RANDOM_STATE = 42
CORR_THRESHOLD = 0.80

RIDGE_ALPHAS = np.logspace(-4, 4, 50)
POLY_DEGREES = (1, 2, 3)
RF_N_ESTIMATORS = (100, 300, 500)
RF_MAX_DEPTHS = (5, 10, 20, None)
RF_MIN_SAMPLES_SPLITS = (2, 5, 10)
RF_MIN_SAMPLES_LEAFS = (1, 2, 4)


## 3. Data preparation and feature selection

In [3]:
def prepare_data(df):
    assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-8

    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    n = len(df)
    n_train = int(TRAIN_SIZE * n)
    n_val = int(VAL_SIZE * n)

    train_df = df.iloc[:n_train, 4:].copy()
    val_df = df.iloc[n_train:n_train + n_val, 4:].copy()
    test_df = df.iloc[n_train + n_val:, 4:].copy()

    train_mean = train_df.mean()
    train_std = train_df.std().replace(0, 1)

    train_df = (train_df - train_mean) / train_std
    val_df = (val_df - train_mean) / train_std
    test_df = (test_df - train_mean) / train_std

    # Preserve the existing dataset convention:
    # six input features immediately before the final target column.
    X_train = train_df.iloc[:, -7:-1]
    y_train = train_df.iloc[:, -1]
    X_val = val_df.iloc[:, -7:-1]
    y_val = val_df.iloc[:, -1]
    X_test = test_df.iloc[:, -7:-1]
    y_test = test_df.iloc[:, -1]

    return X_train, X_val, X_test, y_train, y_val, y_test, train_mean, train_std


def select_features(X_train, y_train, X_val, X_test, threshold=CORR_THRESHOLD):
    ranking = X_train.corrwith(y_train).abs().sort_values(ascending=False)
    selected = []

    for feature in ranking.index:
        if len(selected) == 3:
            break

        if not selected:
            selected.append(feature)
            continue

        correlations = X_train[selected].corrwith(X_train[feature]).abs()

        if (correlations < threshold).all():
            selected.append(feature)

    return (
        X_train[selected],
        X_val[selected],
        X_test[selected],
        selected
    )

## 4. Models

In [4]:
def train_ridge(X_train, y_train, X_val, y_val, X_test, y_test):
    best_model = None
    best_alpha = None
    best_val_r2 = -np.inf

    for alpha in RIDGE_ALPHAS:
        model = Ridge(alpha=alpha)
        model.fit(X_train, y_train)
        val_r2 = r2_score(y_val, model.predict(X_val))

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_alpha = alpha
            best_model = model

    test_r2 = r2_score(y_test, best_model.predict(X_test))
    return best_model, {"Best Alpha": best_alpha, "Validation R²": best_val_r2, "Test R²": test_r2}


def train_polynomial_ridge(X_train, y_train, X_val, y_val, X_test, y_test):
    best_model = None
    best_poly = None
    best_degree = None
    best_alpha = None
    best_val_r2 = -np.inf

    for degree in POLY_DEGREES:
        poly = PolynomialFeatures(degree=degree, include_bias=True)
        Xtr = poly.fit_transform(X_train)
        Xv = poly.transform(X_val)
        Xte = poly.transform(X_test)

        for alpha in RIDGE_ALPHAS:
            model = Ridge(alpha=alpha)
            model.fit(Xtr, y_train)
            val_r2 = r2_score(y_val, model.predict(Xv))

            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                best_degree = degree
                best_alpha = alpha
                best_model = model
                best_poly = poly
                best_Xv = Xv
                best_Xte = Xte

    val_r2 = r2_score(y_val, best_model.predict(best_Xv))
    test_r2 = r2_score(y_test, best_model.predict(best_Xte))

    info = {
        "Best Degree": best_degree,
        "Best Alpha": best_alpha,
        "Validation R²": val_r2,
        "Test R²": test_r2,
    }
    return best_model, best_poly, info


def train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test):
    best_model = None
    best_params = None
    best_val_r2 = -np.inf

    for n_estimators in RF_N_ESTIMATORS:
        for max_depth in RF_MAX_DEPTHS:
            for min_samples_split in RF_MIN_SAMPLES_SPLITS:
                for min_samples_leaf in RF_MIN_SAMPLES_LEAFS:
                    base_model = RandomForestRegressor(
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_split=min_samples_split,
                        min_samples_leaf=min_samples_leaf,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    )
                    model = MultiOutputRegressor(base_model)
                    model.fit(X_train, y_train.to_frame())
                    val_r2 = r2_score(y_val, model.predict(X_val))

                    if val_r2 > best_val_r2:
                        best_val_r2 = val_r2
                        best_model = model
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_split": min_samples_split,
                            "min_samples_leaf": min_samples_leaf,
                        }

    test_r2 = r2_score(y_test, best_model.predict(X_test))
    info = {**best_params, "Validation R²": best_val_r2, "Test R²": test_r2}
    return best_model, info

## 5. Train all systems with the same pipeline

In [ ]:
all_results = []

for system, config in SYSTEMS.items():
    print(f"\n{'=' * 60}\n{system}\n{'=' * 60}")

    df = pd.read_csv(DATA_DIR / system / config["file"])

    (X_train, X_val, X_test,
     y_train, y_val, y_test,
     train_mean, train_std) = prepare_data(df)

    X_train, X_val, X_test, selected = select_features(
        X_train, y_train, X_val, X_test
    )

    print("Selected features:", selected)
    print("Train / validation / test:", X_train.shape, X_val.shape, X_test.shape)

    # Save scaler using the selected features and target.
    scaler_cols = selected + ["Ratio"]
    system_model_dir = MODEL_DIR / system
    system_model_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(
        {"mean": train_mean[scaler_cols], "std": train_std[scaler_cols]},
        system_model_dir / f"{system}_T3b_scaler.pkl",
    )

    # Ridge
    ridge_model, ridge_info = train_ridge(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    joblib.dump(ridge_model, system_model_dir / f"{system}_T3b_linear_model.pkl")

    # Polynomial Ridge
    poly_model, poly_transform, poly_info = train_polynomial_ridge(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    joblib.dump(poly_model, system_model_dir / f"{system}_T3b_poly_model.pkl")
    joblib.dump(poly_transform, system_model_dir / f"{system}_T3b_poly_transform.pkl")

    # Random Forest
    rf_model, rf_info = train_random_forest(
        X_train, y_train, X_val, y_val, X_test, y_test
    )
    joblib.dump(rf_model, system_model_dir / f"{system}_T3b_rf_model.pkl")

    all_results.extend([
        {"System": system, "Model": "Ridge", **ridge_info},
        {"System": system, "Model": "Polynomial Ridge", **poly_info},
        {"System": system, "Model": "Random Forest", **rf_info},
    ])

results_df = pd.DataFrame(all_results)
display(results_df)


NiCoMn
Selected features: ['AA_pct', 'AB_pct', 'BB_pct']
Train / validation / test: (116, 3) (16, 3) (34, 3)

NiFeTi
Selected features: ['CC_pct', 'AB_pct', 'BC_pct']
Train / validation / test: (116, 3) (16, 3) (34, 3)

ZnYGe
Selected features: ['CC_pct', 'BB_pct', 'AA_pct']
Train / validation / test: (116, 3) (16, 3) (34, 3)


,System,Model,Best Alpha,Validation R²,Test R²,Best Degree,n_estimators,max_depth,min_samples_split,min_samples_leaf
0,NiCoMn,Ridge,7.906043,0.592182,0.645658,NaN,NaN,NaN,NaN,NaN
1,NiCoMn,Polynomial Ridge,0.828643,0.907679,0.810086,3.0,NaN,NaN,NaN,NaN
2,NiCoMn,Random Forest,NaN,0.811506,0.699712,NaN,300.0,10.0,2.0,1.0
3,NiFeTi,Ridge,0.000100,0.835605,0.843534,NaN,NaN,NaN,NaN,NaN
4,NiFeTi,Polynomial Ridge,0.000100,0.861578,0.873943,2.0,NaN,NaN,NaN,NaN
5,NiFeTi,Random Forest,NaN,0.847178,0.737186,NaN,100.0,20.0,2.0,1.0
6,ZnYGe,Ridge,2.559548,0.942614,0.847406,NaN,NaN,NaN,NaN,NaN
7,ZnYGe,Polynomial Ridge,2.559548,0.942614,0.847406,1.0,NaN,NaN,NaN,NaN
8,ZnYGe,Random Forest,NaN,0.829543,0.813174,NaN,300.0,10.0,10.0,2.0


: 